In [ ]:
# SPR 2026 - Ensemble Stacking: BERTimbau + mDeBERTa + XLM-R
# Treina 3 modelos diferentes, coleta OOF predictions, e treina
# um meta-learner LightGBM sobre as probabilidades empilhadas.
# Diversidade de arquiteturas e o principal motor de melhoria.

import os
import re
import gc
import time
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

# ==========================================================================
# CONFIG
# ==========================================================================
SEED = 42
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5
N_FOLDS = 5
NUM_CLASSES = 7
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

DATA_DIR = '/kaggle/input/competitions/spr-2026-mammography-report-classification'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

MODELS = {
    'bertimbau': {
        'paths': [
            '/kaggle/input/models/fabianofilho/bertimbau-ptbr-complete/pytorch/default/1',
            '/kaggle/input/bertimbau-ptbr-complete/pytorch/bert-large-portuguese-cased/1',
        ],
        'batch_size': 8,
        'has_token_type_ids': True,
    },
    'mdeberta': {
        'paths': [
            '/kaggle/input/models/microsoft/mdeberta-v3-base/pytorch/default/1',
            '/kaggle/input/mdeberta-v3-base',
        ],
        'batch_size': 16,
        'has_token_type_ids': False,
    },
    'xlmr': {
        'paths': [
            '/kaggle/input/models/FacebookAI/xlm-roberta-base/pytorch/default/1',
            '/kaggle/input/xlm-roberta-base',
        ],
        'batch_size': 16,
        'has_token_type_ids': False,
    },
}


def find_model_path(model_cfg):
    """Tenta cada path listado e retorna o primeiro que existe."""
    for p in model_cfg['paths']:
        if os.path.isdir(p) and os.path.exists(os.path.join(p, 'config.json')):
            return p
    # fallback: busca recursiva a partir do primeiro diretorio pai
    for p in model_cfg['paths']:
        base = os.path.dirname(p)
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if 'config.json' in files:
                return root
    return None


print(f'Device: {DEVICE}')
for name, cfg in MODELS.items():
    path = find_model_path(cfg)
    print(f'{name}: {path}')

In [ ]:
# ==========================================================================
# DATA LOADING + PREPROCESSING
# ==========================================================================
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df = pd.read_csv(f'{DATA_DIR}/test.csv')

INDICACAO_MARKERS = [
    'indica\u00e7\u00e3o cl\u00ednica:\n',
    'indica\u00e7\u00e3o cl\u00ednica:',
    'indica\u00e7\u00e3o:',
    'indicacao:',
]
ACHADOS_MARKERS = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['an\u00e1lise comparativa:', 'analise comparativa:']


def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()


def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def build_input_text(report_text):
    report = clean_text(report_text)
    indicacao = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:
        parts.append(f'Indica\u00e7\u00e3o: {indicacao}')
    if achados:
        parts.append(f'Achados: {achados}')
    if comparativa:
        parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)


train_texts = [build_input_text(t) for t in train_df['report'].tolist()]
test_texts = [build_input_text(t) for t in test_df['report'].tolist()]
train_labels = train_df['target'].values

print(f'Train: {len(train_df)}, Test: {len(test_df)}')
print(train_df['target'].value_counts().sort_index())

In [ ]:
# ==========================================================================
# DATASET + FOCAL LOSS
# ==========================================================================
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len, labels=None, has_token_type_ids=True):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = labels
        self.has_token_type_ids = has_token_type_ids

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if self.has_token_type_ids and 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce)
        return (self.alpha * (1 - pt) ** self.gamma * ce).mean()

In [ ]:
# ==========================================================================
# TRAINING HELPERS (FP16, per-model token_type_ids)
# ==========================================================================
criterion = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)


def train_epoch(model, loader, optimizer, scheduler, scaler, has_token_type_ids):
    model.train()
    total_loss = 0
    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if has_token_type_ids and 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        with autocast():
            outputs = model(**kwargs)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def evaluate(model, loader, has_token_type_ids):
    model.eval()
    all_probs = []
    all_labels = []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if has_token_type_ids and 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        with autocast():
            outputs = model(**kwargs)
        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
        if 'labels' in batch:
            all_labels.append(batch['labels'].numpy())
    all_probs = np.vstack(all_probs)
    if all_labels:
        all_labels = np.concatenate(all_labels)
        return all_probs, all_labels
    return all_probs, None


@torch.no_grad()
def predict(model, loader, has_token_type_ids):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if has_token_type_ids and 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)

        with autocast():
            outputs = model(**kwargs)
        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)


def compute_f1(probs, labels):
    preds = np.argmax(probs, axis=1)
    return f1_score(labels, preds, average='macro')


print('Training helpers prontos.')

In [ ]:
# ==========================================================================
# TRAIN ALL 3 MODELS IN SEQUENCE
# For each model:
#   - 5-fold CV
#   - Collect OOF probs (N_train, 7)
#   - Collect test probs (N_test, 7)
# Stack: oof_stack (N_train, 21), test_stack (N_test, 21)
# ==========================================================================
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(train_texts, train_labels))

oof_dict = {}    # model_name -> (N_train, 7)
test_dict = {}   # model_name -> (N_test, 7)

for model_name, model_cfg in MODELS.items():
    print('\n' + '=' * 60)
    print(f'MODELO: {model_name.upper()}')
    print('=' * 60)

    model_path = find_model_path(model_cfg)
    if model_path is None:
        print(f'AVISO: {model_name} nao encontrado, pulando...')
        continue

    bs = model_cfg['batch_size']
    has_tti = model_cfg['has_token_type_ids']

    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

    oof_probs = np.zeros((len(train_texts), NUM_CLASSES))
    test_probs = np.zeros((len(test_texts), NUM_CLASSES))
    fold_scores = []

    t0_model = time.time()

    for fold_idx, (train_idx, val_idx) in enumerate(folds):
        print(f'\n  Fold {fold_idx + 1}/{N_FOLDS}')
        t0_fold = time.time()

        # Datasets
        fold_train_texts = [train_texts[i] for i in train_idx]
        fold_train_labels = train_labels[train_idx]
        fold_val_texts = [train_texts[i] for i in val_idx]
        fold_val_labels = train_labels[val_idx]

        train_ds = MammographyDataset(
            fold_train_texts, tokenizer, MAX_LEN,
            labels=fold_train_labels, has_token_type_ids=has_tti,
        )
        val_ds = MammographyDataset(
            fold_val_texts, tokenizer, MAX_LEN,
            labels=fold_val_labels, has_token_type_ids=has_tti,
        )
        test_ds = MammographyDataset(
            test_texts, tokenizer, MAX_LEN,
            labels=None, has_token_type_ids=has_tti,
        )

        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=bs * 2, shuffle=False, num_workers=2, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=bs * 2, shuffle=False, num_workers=2, pin_memory=True)

        # Model
        model = AutoModelForSequenceClassification.from_pretrained(
            model_path, num_labels=NUM_CLASSES, local_files_only=True,
        ).to(DEVICE)

        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        total_steps = len(train_loader) * EPOCHS
        warmup_steps = int(total_steps * 0.1)
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
        scaler = GradScaler()

        best_f1 = 0
        best_val_probs = None
        best_test_probs = None

        for epoch in range(EPOCHS):
            loss = train_epoch(model, train_loader, optimizer, scheduler, scaler, has_tti)
            val_p, val_l = evaluate(model, val_loader, has_tti)
            f1 = compute_f1(val_p, val_l)
            print(f'    Epoch {epoch + 1}/{EPOCHS}: loss={loss:.4f}, val_f1={f1:.5f}')

            if f1 > best_f1:
                best_f1 = f1
                best_val_probs = val_p.copy()
                best_test_probs = predict(model, test_loader, has_tti)

        oof_probs[val_idx] = best_val_probs
        test_probs += best_test_probs / N_FOLDS
        fold_scores.append(best_f1)

        elapsed = time.time() - t0_fold
        print(f'    Best F1: {best_f1:.5f} ({elapsed:.0f}s)')

        del model, optimizer, scheduler, scaler
        del train_ds, val_ds, test_ds, train_loader, val_loader, test_loader
        gc.collect()
        torch.cuda.empty_cache()

    oof_dict[model_name] = oof_probs
    test_dict[model_name] = test_probs

    elapsed_model = time.time() - t0_model
    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    oof_f1 = compute_f1(oof_probs, train_labels)
    print(f'\n  {model_name} CV: {mean_f1:.5f} +/- {std_f1:.5f}')
    print(f'  {model_name} OOF F1: {oof_f1:.5f}')
    print(f'  Tempo total: {elapsed_model:.0f}s')

# Stack all OOF and test predictions
model_names = list(oof_dict.keys())
oof_stack = np.hstack([oof_dict[m] for m in model_names])  # (N_train, 7*N_models)
test_stack = np.hstack([test_dict[m] for m in model_names])  # (N_test, 7*N_models)

print(f'\nModelos treinados: {model_names}')
print(f'oof_stack shape: {oof_stack.shape}')
print(f'test_stack shape: {test_stack.shape}')

In [ ]:
# ==========================================================================
# LIGHTGBM META-LEARNER
# Treina LightGBM sobre oof_stack -> train_labels
# 5-fold CV para o meta-learner tambem
# ==========================================================================
print('\n' + '=' * 60)
print('LIGHTGBM META-LEARNER')
print('=' * 60)

lgb_params = {
    'objective': 'multiclass',
    'num_class': NUM_CLASSES,
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 500,
    'random_state': SEED,
    'verbosity': -1,
}

meta_oof_probs = np.zeros((len(train_labels), NUM_CLASSES))
meta_test_probs = np.zeros((len(test_texts), NUM_CLASSES))
meta_fold_scores = []

skf_meta = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + 1)

for fold_idx, (train_idx, val_idx) in enumerate(skf_meta.split(oof_stack, train_labels)):
    print(f'\n  Meta-Fold {fold_idx + 1}/{N_FOLDS}')

    X_tr = oof_stack[train_idx]
    y_tr = train_labels[train_idx]
    X_va = oof_stack[val_idx]
    y_va = train_labels[val_idx]

    lgb_model = lgb.LGBMClassifier(**lgb_params)
    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(0),
        ],
    )

    val_probs = lgb_model.predict_proba(X_va)
    meta_oof_probs[val_idx] = val_probs
    fold_f1 = f1_score(y_va, val_probs.argmax(axis=1), average='macro')
    meta_fold_scores.append(fold_f1)
    print(f'    F1: {fold_f1:.5f} (best_iter: {lgb_model.best_iteration_})')

    meta_test_probs += lgb_model.predict_proba(test_stack) / N_FOLDS

meta_oof_f1 = f1_score(train_labels, meta_oof_probs.argmax(axis=1), average='macro')
print(f'\nMeta-Learner CV: {np.mean(meta_fold_scores):.5f} +/- {np.std(meta_fold_scores):.5f}')
print(f'Meta-Learner OOF F1: {meta_oof_f1:.5f}')

# Comparar com media simples dos modelos base
simple_avg_oof = np.zeros((len(train_labels), NUM_CLASSES))
for m in model_names:
    simple_avg_oof += oof_dict[m]
simple_avg_oof /= len(model_names)
simple_avg_f1 = f1_score(train_labels, simple_avg_oof.argmax(axis=1), average='macro')
print(f'\nComparacao:')
for m in model_names:
    print(f'  {m} OOF F1: {compute_f1(oof_dict[m], train_labels):.5f}')
print(f'  Media simples OOF F1: {simple_avg_f1:.5f}')
print(f'  LightGBM stacking OOF F1: {meta_oof_f1:.5f}')

In [ ]:
# ==========================================================================
# OPTUNA CALIBRATION ON META-LEARNER OOF PREDICTIONS
# ==========================================================================
print('\n' + '=' * 60)
print('OPTUNA CALIBRATION')
print('=' * 60)


def temperature_scale(probs, temperature):
    logits = np.log(probs + 1e-10)
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)


def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)


N_TRIALS = 200


def objective(trial):
    temp = trial.suggest_float('temperature', 0.3, 3.0)
    calibrated = temperature_scale(meta_oof_probs, temp)
    thresholds = [
        trial.suggest_float(f't{c}', 0.05, 3.0) for c in range(NUM_CLASSES)
    ]
    preds = apply_thresholds(calibrated, thresholds)
    return f1_score(train_labels, preds, average='macro')


study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_temp = study.best_params['temperature']
best_thresholds = [study.best_params[f't{c}'] for c in range(NUM_CLASSES)]

print(f'\nMelhor F1 OOF (Optuna): {study.best_value:.5f}')
print(f'Temperatura: {best_temp:.4f}')
print(f'Thresholds: {[round(t, 4) for t in best_thresholds]}')

# F1 sem calibracao vs com calibracao
print(f'\nF1 sem calibracao: {meta_oof_f1:.5f}')
print(f'F1 com calibracao: {study.best_value:.5f}')

In [ ]:
# ==========================================================================
# SUBMISSION
# ==========================================================================
print('\n' + '=' * 60)
print('SUBMISSION')
print('=' * 60)

# Aplicar calibracao nas predicoes de teste
calibrated_test = temperature_scale(meta_test_probs, best_temp)
predictions = apply_thresholds(calibrated_test, best_thresholds)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'target': predictions,
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print(f'\nDistribuicao das predicoes:')
print(submission['target'].value_counts().sort_index())

print(f'\n=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'\nsubmission.csv salvo ({len(submission)} linhas)')